# 02 — 지역 데이터 생성 (대전 · 홍천 · 순천)**역할:** `01`에서 검증한 절차를 **함수 두 개로 정리**하고 세 지역에 동일하게 적용.```inspect_region(지역)  임상도 shp 읽기 → 임상·수종 구성, 갱신년도, AOI 크기 진단build_labels(지역)    위성 tif 격자를 그대로 물려받아 폴리곤을 래스터화```**설계상 중요한 두 가지**1. **AOI를 임상도 경계에서 자동 추출** → 위성 범위와 라벨 범위가 자동 일치 (좌표를 손으로 안 찍음)2. **GEE export 단계에서 `crs='EPSG:5179'` 지정** → 재투영 자체를 제거. 정합 사고의 절반이 사라짐## 최종 보고서에 인용된 셀| 셀 | 내용 | 보고서 ||---|---|---|| `[3]` `[5]` | 지역별 임상·수종 구성 (낙엽송 0 / 8.6 / 33%) | 3절 || `[12]` | **갱신년도 분포** — 2020+ 잔존율 45~92% | 2절 || `[14]` | **낙엽송 가설 검증** — 겨울 NDVI 0.369 vs 활엽 0.341~0.353 | 5절 |실행 순서: `inspect` → `export` → (GEE 완료 대기) → `[8]` tif 이동 → `[11]` 라벨 생성 → `[13]` 정합 검증

# 02 — 지역 데이터 생성 (대전 · 홍천 · 순천)`01`에서 검증한 절차를 **함수 두 개로 묶어** 세 지역에 똑같이 적용합니다.```inspect_region(지역)   임상도를 읽어 임상·수종 구성, 갱신년도, AOI 크기를 진단build_labels(지역)     위성 tif 격자를 그대로 물려받아 폴리곤을 래스터화```**실행 순서**`inspect` → `export` → *(GEE 완료 대기)* → `[8]` tif 이동 → `[11]` 라벨 생성 → `[13]` 정합 검증---## 준비

In [ ]:
# ============================================================
# [1] 셋업
#   지역 데이터 생성 전용 노트북.
#   01_explore 에서 검증한 절차를 함수로 묶어 지역마다 재사용한다.
# ============================================================
!pip install -q earthengine-api geemap geopandas pyogrio rasterio

import ee, geemap, numpy as np, pandas as pd, geopandas as gpd
import rasterio, pyogrio, os, glob, shutil
from rasterio.features import rasterize
from google.colab import drive

PROJECT_ID = 'driven-era-467023-m3'
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT_ID)

drive.mount('/content/drive')
D = '/content/drive/MyDrive/forest/data'

BANDS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
CLASS_MAP = {'0':0, '1':1, '2':2, '3':3, '4':0}   # 죽림은 배경 흡수

def cloud_mask(img):
    scl = img.select('SCL')
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good).divide(10000)

def composite_aoi(aoi, start, end, months, cloud_pct=50):
    mf = ee.Filter.Or([ee.Filter.calendarRange(m, m, 'month') for m in months])
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi).filterDate(start, end).filter(mf)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_pct))
            .map(cloud_mask).median().clip(aoi))

print('셋업 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.3 MB/s eta 0:00:00
Mounted at /content/drive
셋업 완료


---## 공통 함수

In [ ]:
# ============================================================
# [2] 지역 데이터 생성 함수
#   - inspect_region : 임상도를 읽고 진단 정보 출력 (export 전 필수 확인)
#   - export_region  : AOI를 잘라 20밴드 스택을 Drive로 내보냄
#       max_km : AOI 한 변의 최대 길이(km). 지역 간 면적을 맞춰
#                비교 공정성을 확보하고 Colab RAM 초과를 방지
#       int16  : 반사율을 10000배 정수로 저장 → 파일 크기 절반
#                (읽을 때 /10000 필요)
# ============================================================
def inspect_region(name, shp_dir):
    shp  = glob.glob(f'{D}/imsang/{shp_dir}/*.shp')[0]
    info = pyogrio.read_info(shp)
    print(f'=== {name} ===')
    print('파일   :', os.path.basename(shp))
    print('좌표계 :', info['crs'])
    print('폴리곤 :', f"{info['features']:,}")

    gdf = gpd.read_file(shp, engine='pyogrio')
    gdf['area_ha'] = gdf.geometry.area / 10000

    print('\n임상 면적')
    t = gdf.groupby(['FRTP_CD','FRTP_NM'])['area_ha'].sum()
    for (cd, nm), a in t.items():
        print(f'  {nm:12s} {a:9,.0f} ha  ({a*100/t.sum():5.1f}%)')

    yr = pd.to_numeric(gdf['갱신년도'], errors='coerce')
    print('\n갱신년도 :', yr.value_counts().head(6).to_dict())
    print(f'2024년 이후 : {(yr>=2024).sum()*100/len(yr):.1f}%')

    # 상록활엽수 확인 (순천에서 특히 중요)
    if 'KOFTR_GROU' in gdf.columns:
        ev = pd.to_numeric(gdf['KOFTR_GROU'], errors='coerce')
        n_ev = ((ev >= 61) & (ev <= 68)).sum()
        print(f'상록활엽수 폴리곤 : {n_ev:,}개 ({n_ev*100/len(gdf):.1f}%)')

    b = gdf.to_crs(4326).total_bounds
    print(f'경계 크기 : {(b[2]-b[0])*88:.0f} x {(b[3]-b[1])*111:.0f} km')
    return gdf, info['crs']


def export_region(name, gdf, crs, start='2024-01-01', end='2025-12-31',
                  max_km=35, int16=True):
    b = gdf.to_crs(4326).total_bounds
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    dx = min((b[2]-b[0])/2, max_km/2/88)      # 경도 1도 ≈ 88km (위도 36~38도)
    dy = min((b[3]-b[1])/2, max_km/2/111)     # 위도 1도 ≈ 111km
    aoi = ee.Geometry.Rectangle([cx-dx, cy-dy, cx+dx, cy+dy])
    print(f'{name} AOI 사용 : {dx*2*88:.0f} x {dy*2*111:.0f} km')

    summer  = composite_aoi(aoi, start, end, [6,7,8,9])
    leafoff = composite_aoi(aoi, start, end, [11,12,1,2])

    for nm, mo in [('여름',[6,7,8,9]), ('낙엽기',[11,12,1,2])]:
        mf = ee.Filter.Or([ee.Filter.calendarRange(m,m,'month') for m in mo])
        n = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterBounds(aoi).filterDate(start,end).filter(mf)
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',50)).size().getInfo())
        print(f'  {nm}: {n}장')

    stack = (summer.select(BANDS).rename([f's_{b}' for b in BANDS])
             .addBands(leafoff.select(BANDS).rename([f'w_{b}' for b in BANDS])))
    stack = stack.multiply(10000).toInt16() if int16 else stack.toFloat()

    task = ee.batch.Export.image.toDrive(
        image=stack, description=f'{name}_s2',
        folder='forest_s2', fileNamePrefix=f'{name}_s2_20band',
        region=aoi, scale=10, crs=str(crs), maxPixels=1e10)
    task.start()
    print('  export 제출' + ('  [int16, 읽을 때 /10000]' if int16 else ''))
    return task, aoi

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


---## 지역별 진단과 export

In [ ]:
# [3] 홍천 진단
gdf_hc, crs_hc = inspect_region('hongcheon', '2025_hongcheon')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


=== hongcheon ===
파일   : 51720.shp
좌표계 : EPSG:5179
폴리곤 : 76,223

임상 면적
  무립목지/비산림         2,061 ha  (  1.4%)
  침엽수림            54,219 ha  ( 35.7%)
  활엽수림            78,223 ha  ( 51.5%)
  혼효림             17,326 ha  ( 11.4%)

갱신년도 : {2025.0: 15532, 2024.0: 11679, 2017.0: 11329, 2019.0: 8559, 2015.0: 4178, 2021.0: 3723}
2024년 이후 : 35.7%
상록활엽수 폴리곤 : 0개 (0.0%)
경계 크기 : 93 x 44 km


### `[3-2]` 수종 구성 비교임상 비율이 비슷해도 실제 수종이 다르면 "같은 라벨, 다른 분광"이 됩니다. 이게 전이 실패의 씨앗입니다.

In [ ]:
# ============================================================
# [3-2] 수종 구성 비교
#   임상 비율이 비슷해도 실제 수종이 다르면
#   "같은 라벨, 다른 실체"가 되어 일반화 검증에 의미가 생긴다
# ============================================================
def species_top(gdf, name, n=8):
    g = gdf.copy()
    g['area_ha'] = g.geometry.area / 10000
    t = (g.groupby(['KOFTR_GROU','KOFTR_NM'])['area_ha']
           .sum().sort_values(ascending=False))
    print(f'\n=== {name} 우점 수종 ===')
    for (cd, nm), a in t.head(n).items():
        print(f'  {str(nm)[:14]:16s} {a:8,.0f} ha  ({a*100/t.sum():5.1f}%)')

species_top(gdf_hc, 'hongcheon')

# 대전과 비교하려면 (01_explore 의 gdf 가 아니라 여기서 다시 읽기)
gdf_dj, _ = inspect_region('daejeon', '2025_daejeon')
species_top(gdf_dj, 'daejeon')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]



=== hongcheon 우점 수종 ===
  기타활엽수              29,554 ha  ( 19.5%)
  신갈나무               18,745 ha  ( 12.3%)
  소나무                18,602 ha  ( 12.3%)
  낙엽송                17,982 ha  ( 11.8%)
  기타참나무류             17,974 ha  ( 11.8%)
  침활혼효림              17,326 ha  ( 11.4%)
  잣나무                16,633 ha  ( 11.0%)
  굴참나무                6,873 ha  (  4.5%)
=== daejeon ===
파일   : 30.shp
좌표계 : EPSG:5179
폴리곤 : 18,908

임상 면적
  무립목지/비산림         1,080 ha  (  4.0%)
  침엽수림             8,901 ha  ( 32.8%)
  활엽수림            12,518 ha  ( 46.1%)
  혼효림              4,633 ha  ( 17.1%)
  죽림                  18 ha  (  0.1%)

갱신년도 : {2025.0: 5641, 2015.0: 4883, 2024.0: 4193, 2020.0: 908, 2016.0: 651, 2018.0: 401}
2024년 이후 : 52.0%
상록활엽수 폴리곤 : 0개 (0.0%)
경계 크기 : 28 x 35 km

=== daejeon 우점 수종 ===
  리기다소나무              5,202 ha  ( 19.2%)
  침활혼효림               4,633 ha  ( 17.1%)
  기타참나무류              4,456 ha  ( 16.4%)
  기타활엽수               3,230 ha  ( 11.9%)
  소나무                 2,331 ha  (  8.6%)
  상수리나무        

In [ ]:
# ============================================================
# [4] 홍천 export — 행정구역 전체
#   max_km=999 로 두면 자르지 않고 임상도 경계 전체를 사용
#   int16 저장이라 파일 약 1.5GB 예상 (읽을 때 /10000 필수)
# ============================================================
task_hc, aoi_hc = export_region('hongcheon', gdf_hc, crs_hc, max_km=999)

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


hongcheon AOI 사용 : 93 x 44 km
  여름: 127장
  낙엽기: 173장
  export 제출  [int16, 읽을 때 /10000]


In [ ]:
# ============================================================
# [5] 순천 진단
# ============================================================
gdf_sc, crs_sc = inspect_region('suncheon', '2025_suncheon')
species_top(gdf_sc, 'suncheon')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


=== suncheon ===
파일   : 12150.shp
좌표계 : EPSG:5179
폴리곤 : 27,397

임상 면적
  무립목지/비산림         1,890 ha  (  3.2%)
  침엽수림            28,089 ha  ( 47.6%)
  활엽수림            22,486 ha  ( 38.1%)
  혼효림              5,827 ha  (  9.9%)
  죽림                 659 ha  (  1.1%)

갱신년도 : {2020.0: 8651, 2025.0: 8146, 2021.0: 4953, 2024.0: 1980, 2018.0: 652, 2016.0: 305}
2024년 이후 : 37.0%
상록활엽수 폴리곤 : 70개 (0.3%)
경계 크기 : 36 x 39 km

=== suncheon 우점 수종 ===
  소나무                13,998 ha  ( 23.7%)
  기타참나무류              9,077 ha  ( 15.4%)
  기타활엽수               7,160 ha  ( 12.1%)
  편백나무                6,911 ha  ( 11.7%)
  침활혼효림               5,827 ha  (  9.9%)
  리기다소나무              4,584 ha  (  7.8%)
  밤나무                 3,709 ha  (  6.3%)
  곰솔                  1,902 ha  (  3.2%)


In [ ]:
# ============================================================
# [6] 순천 export
# ============================================================
task_sc, aoi_sc = export_region('suncheon', gdf_sc, crs_sc, max_km=999)

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


suncheon AOI 사용 : 36 x 39 km
  여름: 100장
  낙엽기: 125장
  export 제출  [int16, 읽을 때 /10000]


In [ ]:
# ============================================================
# [7] export 상태 (수시로)
# ============================================================
for nm, t in [('홍천', task_hc), ('순천', task_sc)]:
    print(f'{nm}: {t.status()["state"]}')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


홍천: COMPLETED
순천: COMPLETED


In [ ]:
# ============================================================
# [8] export 결과를 프로젝트 폴더로 이동
#   GEE는 Drive 최상위 forest_s2/ 에만 저장하므로 수동 정리 필요
#   완료된 파일만 옮기고, 아직 없으면 건너뛴다
# ============================================================
import shutil, glob

SRC_DIR = '/content/drive/MyDrive/forest_s2'
DST_DIR = f'{D}/s2'
os.makedirs(DST_DIR, exist_ok=True)

for p in sorted(glob.glob(f'{SRC_DIR}/*.tif')):
    dst = f'{DST_DIR}/{os.path.basename(p)}'
    shutil.move(p, dst)
    print(f'{os.path.basename(dst):35s} {os.path.getsize(dst)/1e6:7.0f} MB')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


suncheon_s2_20band.tif                  579 MB


---## 라벨 생성위성 tif의 격자(`transform`/`width`/`height`)를 그대로 물려받아야픽셀이 어긋나지 않습니다.

In [ ]:
# ============================================================
# [9] 영상 메타 확인
#   좌표계/밴드수/크기가 대전과 동일한 규격인지 점검
# ============================================================
for name in ['daejeon', 'hongcheon', 'suncheon']:
    p = f'{D}/s2/{name}_s2_20band.tif'
    if not os.path.exists(p):
        print(f'{name:10s} (아직 없음)'); continue
    with rasterio.open(p) as s:
        print(f'{name:10s} {s.crs} | {s.count}밴드 | '
              f'{s.width}x{s.height} | {s.dtypes[0]}')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


daejeon    EPSG:5179 | 20밴드 | 2813x3506 | float32
hongcheon  EPSG:5179 | 20밴드 | 9309x4420 | int16
suncheon   (아직 없음)


In [ ]:
# ============================================================
# [10] 임상도 → 라벨 래스터 (지역 공통 함수)
#   위성 tif의 격자를 그대로 물려받아 폴리곤을 굽는다.
#   label : 0=배경 1=침엽 2=활엽 3=혼효 255=nodata
#   year  : 폴리곤 갱신년도 (라벨 시의성 실험용)
#   sp    : 수종그룹코드 (낙엽송/상록활엽수 분석용)  ← 신규
# ============================================================
def build_labels(name, gdf):
    s2 = f'{D}/s2/{name}_s2_20band.tif'
    with rasterio.open(s2) as src:
        meta, transform = src.meta.copy(), src.transform
        shape, s2_crs   = (src.height, src.width), src.crs

    g = gdf.to_crs(s2_crs) if gdf.crs != s2_crs else gdf.copy()

    g['cls'] = g['FRTP_CD'].map(CLASS_MAP).fillna(0).astype('uint8')
    g['yr']  = pd.to_numeric(g['갱신년도'],   errors='coerce').fillna(0).astype('uint16')
    g['sp']  = pd.to_numeric(g['KOFTR_GROU'], errors='coerce').fillna(0).astype('uint16')

    out = {}
    for col, fill, dt in [('cls', 255, 'uint8'),
                          ('yr',    0, 'uint16'),
                          ('sp',    0, 'uint16')]:
        arr = rasterize(((geom, v) for geom, v in zip(g.geometry, g[col])),
                        out_shape=shape, transform=transform,
                        fill=fill, dtype=dt)
        m = meta.copy()
        m.update(count=1, dtype=dt, nodata=fill, compress='lzw')
        path = f'{D}/{name}_{ {"cls":"label","yr":"year","sp":"species"}[col] }.tif'
        with rasterio.open(path, 'w', **m) as dst:
            dst.write(arr, 1)
        out[col] = arr
        print(f'  저장 {os.path.basename(path)}')

    lab = out['cls']
    names_ = {0:'배경', 1:'침엽', 2:'활엽', 3:'혼효', 255:'nodata'}
    print(f'\n{name} 라벨 분포  (격자 {shape[0]}x{shape[1]})')
    u, c = np.unique(lab, return_counts=True)
    for v, n in zip(u, c):
        print(f'  {names_.get(v, v):8s} {n:>12,} ({n*100/lab.size:5.1f}%)')
    return out

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


In [ ]:
# ============================================================
# [11] 라벨 생성 실행
#   export 완료 + tif 이동이 끝난 지역만 처리
# ============================================================
for name, g in [('hongcheon', gdf_hc), ('suncheon', gdf_sc)]:
    if os.path.exists(f'{D}/s2/{name}_s2_20band.tif'):
        build_labels(name, g)
    else:
        print(f'{name}: 위성 tif 아직 없음, 건너뜀')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


  저장 hongcheon_label.tif
  저장 hongcheon_year.tif
  저장 hongcheon_species.tif

hongcheon 라벨 분포  (격자 4420x9309)
  배경            206,085 (  0.5%)
  침엽          5,421,950 ( 13.2%)
  활엽          7,822,581 ( 19.0%)
  혼효          1,732,417 (  4.2%)
  nodata     25,962,747 ( 63.1%)
  저장 suncheon_label.tif
  저장 suncheon_year.tif
  저장 suncheon_species.tif

suncheon 라벨 분포  (격자 3924x3758)
  배경            254,818 (  1.7%)
  침엽          2,808,776 ( 19.0%)
  활엽          2,248,603 ( 15.2%)
  혼효            582,741 (  4.0%)
  nodata      8,851,454 ( 60.0%)


In [ ]:
# ============================================================
# [11-2] 대전 수종 래스터 보강
#   대전은 build_labels 도입 전에 만들어져 species.tif 가 없다.
#   같은 함수로 다시 만들어 세 지역 규격을 통일한다.
#   (label/year 는 동일하게 재생성되므로 덮어써도 무방)
# ============================================================
gdf_dj, crs_dj = inspect_region('daejeon', '2025_daejeon')
build_labels('daejeon', gdf_dj)

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


=== daejeon ===
파일   : 30.shp
좌표계 : EPSG:5179
폴리곤 : 18,908

임상 면적
  무립목지/비산림         1,080 ha  (  4.0%)
  침엽수림             8,901 ha  ( 32.8%)
  활엽수림            12,518 ha  ( 46.1%)
  혼효림              4,633 ha  ( 17.1%)
  죽림                  18 ha  (  0.1%)

갱신년도 : {2025.0: 5641, 2015.0: 4883, 2024.0: 4193, 2020.0: 908, 2016.0: 651, 2018.0: 401}
2024년 이후 : 52.0%
상록활엽수 폴리곤 : 0개 (0.0%)
경계 크기 : 28 x 35 km
  저장 daejeon_label.tif
  저장 daejeon_year.tif
  저장 daejeon_species.tif

daejeon 라벨 분포  (격자 3506x2813)
  배경            109,617 (  1.1%)
  침엽            889,889 (  9.0%)
  활엽          1,252,252 ( 12.7%)
  혼효            463,085 (  4.7%)
  nodata      7,147,535 ( 72.5%)


{'cls': array([[255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        ...,
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255]], dtype=uint8),
 'yr': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=uint16),
 'sp': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=uint16)}

### `[12]` 갱신년도 분포 ★ — 연구 동기순천은 2020년 기준 92%가 최신인데 대전·홍천은 45~51%. **"낡았다"가 아니라 "얼마나 낡았는지가 지역마다 다르다"**가 핵심입니다.

### `[12]` 갱신년도 분포 — 이 연구의 출발점임상도가 **낡았다**가 아니라 **얼마나 낡았는지가 지역마다 다르다**는 것이 핵심입니다.갱신이 밀린 지역이 있는 한 "학습 지역 ≠ 적용 지역" 상황은 피할 수 없습니다.

In [ ]:
# ============================================================
# [12] 학습 가능 픽셀 수 확인
#   갱신년도 필터 기준을 2020+ 로 할지 2024+ 로 할지 판단용.
#   - 지역별로 다른 기준을 쓰면 "지역 차이"와 "라벨 품질 차이"가
#     섞이므로, 세 지역 모두 같은 기준으로 통일해야 한다
#   - 학습 20만 + 평가 30만이 필요하므로 지역당 50만 이상이면 충분
#   아직 라벨을 만들지 않은 지역은 자동으로 건너뛴다
# ============================================================
for name in ['daejeon', 'hongcheon', 'suncheon']:
    lp, yp = f'{D}/{name}_label.tif', f'{D}/{name}_year.tif'
    if not (os.path.exists(lp) and os.path.exists(yp)):
        print(f'{name:10s} (라벨 미생성)')
        continue
    with rasterio.open(lp) as s: lab = s.read(1)
    with rasterio.open(yp) as s: yr  = s.read(1)
    v = (lab >= 1) & (lab <= 3)
    print(f'{name:10s} 전체 {v.sum():>10,} | '
          f'2020+ {(v & (yr>=2020)).sum():>10,} | '
          f'2024+ {(v & (yr>=2024)).sum():>10,}')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


daejeon    전체  2,605,226 | 2020+  1,325,313 | 2024+  1,029,600
hongcheon  전체 14,976,948 | 2020+  6,804,342 | 2024+  4,737,009
suncheon   전체  5,640,120 | 2020+  5,187,176 | 2024+  1,867,787


### `[13]` 정합 검증위성 RGB 위에 라벨을 겹쳐 어긋남을 확인. 좌표계를 export 단계에서 고정했기 때문에 통과했습니다.

---## 검증

In [ ]:
# ============================================================
# [13] 정합 검증 (지역 공통)
#   위성 RGB 위에 라벨을 반투명으로 겹쳐 어긋남을 눈으로 확인.
#   라벨 경계가 산 능선/계곡을 따라가면 정상.
#   int16 저장 지역(홍천/순천)은 /10000 필요.
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from rasterio.windows import Window

def check_align(name, sz=1200):
    s2 = f'{D}/s2/{name}_s2_20band.tif'
    with rasterio.open(f'{D}/{name}_label.tif') as s:
        label = s.read(1)

    # 라벨이 몰려 있는 곳 중심으로 잘라내기
    v = np.argwhere(label != 255)
    cy, cx = v.mean(axis=0).astype(int)
    r0 = max(0, min(cy - sz//2, label.shape[0] - sz))
    c0 = max(0, min(cx - sz//2, label.shape[1] - sz))

    with rasterio.open(s2) as src:
        rgb = src.read([3, 2, 1], window=Window(c0, r0, sz, sz)).astype('float32')
        if src.dtypes[0] == 'int16':
            rgb /= 10000.0
    rgb = np.clip((rgb - 0.02) / 0.23, 0, 1).transpose(1, 2, 0)

    lab   = label[r0:r0+sz, c0:c0+sz]
    lab_m = np.ma.masked_where(lab == 255, lab)
    cmap  = ListedColormap(['#bbbbbb', '#1b7837', '#d95f02', '#e7c419'])

    fig, ax = plt.subplots(1, 3, figsize=(21, 7))
    ax[0].imshow(rgb);                              ax[0].set_title(f'{name} RGB')
    ax[1].imshow(lab_m, cmap=cmap, vmin=0, vmax=3); ax[1].set_title('label')
    ax[2].imshow(rgb)
    ax[2].imshow(lab_m, cmap=cmap, vmin=0, vmax=3, alpha=0.45)
    ax[2].set_title('overlay | 회색=배경 녹색=침엽 주황=활엽 노랑=혼효')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

check_align('suncheon')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


/tmp/ipykernel_419/216601409.py:39: UserWarning: Glyph 54924 (\N{HANGUL SYLLABLE HOE}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipykernel_419/216601409.py:39: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipykernel_419/216601409.py:39: UserWarning: Glyph 48176 (\N{HANGUL SYLLABLE BAE}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipykernel_419/216601409.py:39: UserWarning: Glyph 44221 (\N{HANGUL SYLLABLE GYEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipykernel_419/216601409.py:39: UserWarning: Glyph 45433 (\N{HANGUL SYLLABLE NOG}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipykernel_419/216601409.py:39: UserWarning: Glyph 52840 (\N{HANGUL SYLLABLE CIM}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipykernel_419/216601409.py:39: UserWarning: Glyph 50685 (\N{HANGUL 

<Figure size 2100x700 with 3 Axes>

[그림/지도 출력 생략 — 파일 경량화]


### `[14]` 낙엽송 가설 검증 ★낙엽송(코드 13)은 침엽수인데 **낙엽성**입니다. 겨울 NDVI 0.369로 신갈나무(0.353)·기타활엽수(0.341)와 사실상 구분이 안 됩니다."겨울에 잎 남으면 침엽"을 배운 모델은 낙엽송을 활엽으로 떨구고, 활엽 계수가 크므로 **탄소를 과대추정**합니다.

### `[14]` 낙엽송 확인낙엽송(코드 13)은 침엽수인데 **낙엽성**입니다.낙엽기 NDVI가 활엽수와 겹치면, "겨울에 잎이 남으면 침엽"을 배운 모델은낙엽송을 활엽으로 떨구고 **탄소를 과대추정**합니다.

In [ ]:
# ============================================================
# [14] 낙엽송 가설 검증
#   낙엽송(코드 13)은 침엽수이나 낙엽성이므로,
#   낙엽기 NDVI가 다른 침엽수보다 낮고 활엽수에 가까울 것이다.
#   이것이 확인되면 "침엽=상록" 전제의 예외를 정량화한 것이 된다.
# ============================================================
with rasterio.open(f'{D}/hongcheon_species.tif') as s: sp  = s.read(1)
with rasterio.open(f'{D}/hongcheon_label.tif')   as s: lab = s.read(1)

with rasterio.open(f'{D}/s2/hongcheon_s2_20band.tif') as s:
    w_nir = s.read(17).astype('float32') / 10000
    w_red = s.read(13).astype('float32') / 10000
w_ndvi = (w_nir - w_red) / (w_nir + w_red + 1e-6)

targets = [(11,'소나무'), (12,'잣나무'), (13,'낙엽송'),
           (32,'신갈나무'), (30,'기타활엽수')]
print('수종별 낙엽기 NDVI 중앙값')
for cd, nm in targets:
    m = (sp == cd) & (lab >= 1) & (lab <= 3)
    if m.sum() > 1000:
        print(f'  {nm:10s} {np.nanmedian(w_ndvi[m]):.3f}   ({m.sum():>9,} px)')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


수종별 낙엽기 NDVI 중앙값
  소나무        0.632   (1,860,034 px)
  잣나무        0.685   (1,663,610 px)
  낙엽송        0.369   (1,798,157 px)
  신갈나무       0.353   (1,874,298 px)
  기타활엽수      0.341   (2,955,886 px)
